# Chapter 9.6: Training Frameworks for Recommendation Systems

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** TorchRec's architecture for sharded embeddings, planning, and distributed training
2. **Describe** HugeCTR's GPU-native approach to recommendation training
3. **Compare** DeepRec, Merlin, and XDL in terms of features, performance, and ecosystem
4. **Implement** TorchRec-style embedding sharding and planning logic
5. **Build** a simple rec training pipeline using framework-inspired patterns
6. **Evaluate** framework trade-offs for different organizational contexts
7. **Design** a framework selection strategy based on team and model requirements

## Prerequisites

- Understanding of distributed training concepts (Chapter 9.2)
- PyTorch experience
- Familiarity with recommendation model architectures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part9/chapter_9.6_training_frameworks.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://github.com/USERNAME/rec_system/blob/main/notebooks/part9/chapter_9.6_training_frameworks.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import time
from typing import List, Dict, Tuple, Optional
from collections import defaultdict
from dataclasses import dataclass, field

np.random.seed(42)
torch.manual_seed(42)

print("All imports successful!")
print(f"PyTorch version: {torch.__version__}")

## 1. The Landscape of RecSys Training Frameworks

| Framework | Company | Backend | Key Feature | Scale | Open Source |
|-----------|---------|---------|-------------|-------|-------------|
| **TorchRec** | Meta | PyTorch | Sharded embeddings, planner | TBs | Yes (2022) |
| **HugeCTR** | NVIDIA | CUDA/C++ | GPU-native, high throughput | TBs | Yes (2020) |
| **DeepRec** | Alibaba | TensorFlow | Rec-specific TF ops | TBs | Yes (2022) |
| **Merlin** | NVIDIA | Multi | End-to-end pipeline | GBs-TBs | Yes (2021) |
| **XDL** | Alibaba | Custom | eXtreme scale | PBs | Partially (2019) |
| **Persia** | Kuaishou | PyTorch | Hybrid CPU-GPU training | 100T params | Yes (2022) |

> **💡 Concept:** The choice of framework depends on your existing stack (TF vs PyTorch), scale requirements, team expertise, and the specific features needed (e.g., online learning, feature engineering, serving integration).

## 2. TorchRec (Meta, 2022)

TorchRec is Meta's PyTorch-native library for building recommendation systems at scale. It powers all recommendation models at Meta (Facebook, Instagram, etc.).

### Core Abstractions

1. **`EmbeddingBagCollection`**: Collection of embedding tables with pooling
2. **`ShardedModule`**: Wraps modules with automatic sharding across devices
3. **`EmbeddingShardingPlanner`**: Automatically determines optimal sharding strategy
4. **`DistributedModelParallel`**: Wraps a model for distributed training

### Sharding Strategies in TorchRec

- **Table-wise**: One table per device
- **Row-wise**: Rows split across devices  
- **Column-wise**: Embedding dimensions split
- **Table-wise + Row-wise**: Hybrid
- **Data-parallel**: Small tables replicated on all devices

The planner solves a **constrained optimization** problem:

$$\min_{\text{plan}} \max_{g \in \text{GPUs}} \text{cost}(g; \text{plan})$$
$$\text{s.t.} \quad \text{memory}(g) \leq M_g \quad \forall g$$

In [ ]:
# Simulate TorchRec-style abstractions

@dataclass
class EmbeddingTableConfig:
    """Configuration for a single embedding table (TorchRec-style)."""
    name: str
    num_embeddings: int
    embedding_dim: int
    feature_names: List[str] = field(default_factory=list)
    pooling: str = 'sum'  # sum, mean, none
    
    @property
    def memory_bytes(self) -> int:
        return self.num_embeddings * self.embedding_dim * 4  # FP32
    
    @property
    def memory_mb(self) -> float:
        return self.memory_bytes / (1024 ** 2)


@dataclass
class ShardingPlan:
    """Describes how to shard embedding tables across devices."""
    table_name: str
    strategy: str  # 'table_wise', 'row_wise', 'data_parallel'
    device_ids: List[int] = field(default_factory=list)
    rows_per_device: Optional[int] = None


class EmbeddingBagCollection(nn.Module):
    """Simulated TorchRec-style EmbeddingBagCollection."""
    
    def __init__(self, tables: List[EmbeddingTableConfig]):
        super().__init__()
        self.table_configs = {t.name: t for t in tables}
        self.embeddings = nn.ModuleDict()
        
        for table in tables:
            if table.pooling == 'none':
                self.embeddings[table.name] = nn.Embedding(
                    table.num_embeddings, table.embedding_dim
                )
            else:
                self.embeddings[table.name] = nn.EmbeddingBag(
                    table.num_embeddings, table.embedding_dim,
                    mode=table.pooling
                )
            nn.init.xavier_uniform_(self.embeddings[table.name].weight)
    
    def forward(self, features: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        """Look up embeddings for all features."""
        outputs = {}
        for name, indices in features.items():
            if name in self.embeddings:
                table_config = self.table_configs[name]
                if table_config.pooling == 'none':
                    outputs[name] = self.embeddings[name](indices)
                else:
                    # For EmbeddingBag, need offsets
                    if indices.dim() == 1:
                        outputs[name] = self.embeddings[name](indices, 
                            torch.arange(0, len(indices) + 1, dtype=torch.long)[:-1])
                    else:
                        # Flatten and create offsets
                        flat = indices.reshape(-1)
                        offsets = torch.arange(0, flat.size(0), indices.size(1))
                        outputs[name] = self.embeddings[name](flat, offsets)
        return outputs
    
    def total_memory_mb(self) -> float:
        return sum(t.memory_mb for t in self.table_configs.values())


# Define a DLRM-like model's embedding tables
tables = [
    EmbeddingTableConfig('user_id', 100000, 64, ['user_id'], 'none'),
    EmbeddingTableConfig('item_id', 500000, 64, ['item_id'], 'none'),
    EmbeddingTableConfig('category', 10000, 32, ['category'], 'none'),
    EmbeddingTableConfig('brand', 5000, 32, ['brand'], 'none'),
    EmbeddingTableConfig('user_history', 500000, 64, ['history_items'], 'sum'),
    EmbeddingTableConfig('city', 1000, 16, ['city'], 'none'),
]

ebc = EmbeddingBagCollection(tables)
print(f"Total embedding memory: {ebc.total_memory_mb():.1f} MB")
for name, config in ebc.table_configs.items():
    print(f"  {name}: {config.num_embeddings:>10,} x {config.embedding_dim} = {config.memory_mb:.1f} MB")

In [ ]:
class EmbeddingShardingPlanner:
    """
    Simulated TorchRec-style sharding planner.
    Determines optimal sharding strategy for each embedding table.
    """
    
    def __init__(self, num_devices: int, device_memory_mb: float,
                 replicate_threshold_mb: float = 10.0):
        self.num_devices = num_devices
        self.device_memory_mb = device_memory_mb
        self.replicate_threshold_mb = replicate_threshold_mb
    
    def plan(self, table_configs: List[EmbeddingTableConfig]) -> List[ShardingPlan]:
        """Generate sharding plan for all tables."""
        plans = []
        device_usage = [0.0] * self.num_devices  # MB used per device
        
        # Sort tables by size (largest first) for better bin-packing
        sorted_tables = sorted(table_configs, key=lambda t: t.memory_mb, reverse=True)
        
        for table in sorted_tables:
            if table.memory_mb < self.replicate_threshold_mb:
                # Small table: replicate on all devices (data parallel)
                plan = ShardingPlan(
                    table_name=table.name,
                    strategy='data_parallel',
                    device_ids=list(range(self.num_devices))
                )
                for d in range(self.num_devices):
                    device_usage[d] += table.memory_mb
            elif table.memory_mb > self.device_memory_mb * 0.5:
                # Very large table: row-wise sharding
                rows_per_device = (table.num_embeddings + self.num_devices - 1) // self.num_devices
                plan = ShardingPlan(
                    table_name=table.name,
                    strategy='row_wise',
                    device_ids=list(range(self.num_devices)),
                    rows_per_device=rows_per_device
                )
                per_device_mb = table.memory_mb / self.num_devices
                for d in range(self.num_devices):
                    device_usage[d] += per_device_mb
            else:
                # Medium table: table-wise (place on least loaded device)
                target_device = min(range(self.num_devices), key=lambda d: device_usage[d])
                plan = ShardingPlan(
                    table_name=table.name,
                    strategy='table_wise',
                    device_ids=[target_device]
                )
                device_usage[target_device] += table.memory_mb
            
            plans.append(plan)
        
        return plans, device_usage


# Plan sharding for our tables
planner = EmbeddingShardingPlanner(num_devices=4, device_memory_mb=16000)
plans, usage = planner.plan(tables)

print("Sharding Plan:")
print(f"{'Table':<20} {'Strategy':<15} {'Devices':<20}")
print("-" * 55)
for plan in plans:
    print(f"{plan.table_name:<20} {plan.strategy:<15} {plan.device_ids}")

print(f"\nDevice Memory Usage:")
for d, u in enumerate(usage):
    print(f"  Device {d}: {u:.1f} MB")
print(f"  Imbalance: {max(usage)/max(min(usage), 0.01):.2f}x")

## 3. HugeCTR (NVIDIA, 2020)

HugeCTR is NVIDIA's GPU-accelerated training framework designed specifically for CTR prediction models.

### Key Design Principles

1. **GPU-native**: Embedding lookups, gradient aggregation, and optimization all happen on GPU
2. **Hierarchical Parameter Server**: GPU HBM → CPU DRAM → SSD for embedding storage
3. **Overlapped Pipeline**: Data loading, embedding lookup, forward, backward all overlapped
4. **Mixed-Precision Native**: FP16 training with automatic loss scaling

### Architecture

```
Data Reader → GPU Embedding Cache → Embedding Lookup → Dense Network → Loss
     ↕              ↕
  Parquet/      CPU Embedding
  Binary          Store (PS)
```

> **🔑 Pro Tip:** HugeCTR can be 5-10x faster than TensorFlow-based approaches for CTR models because it avoids the Python GIL and framework overhead. However, it has a steeper learning curve and less flexibility for custom model architectures.

In [ ]:
class HugeCTRStyleTrainer:
    """
    Simulates HugeCTR's pipelined training approach.
    Overlaps data loading, embedding lookup, and compute.
    """
    
    def __init__(self, model: nn.Module, batch_size: int = 4096,
                 prefetch_batches: int = 3):
        self.model = model
        self.batch_size = batch_size
        self.prefetch_batches = prefetch_batches
        self.optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        
        # Pipeline state
        self.data_queue = []
        self.embedding_cache = {}  # Simulated GPU cache
        
        # Metrics
        self.step_times = []
    
    def _generate_batch(self, num_features: int, num_items: int) -> Dict:
        """Simulate data reader."""
        return {
            'features': torch.randint(0, num_items, (self.batch_size, num_features)),
            'labels': torch.randint(0, 2, (self.batch_size,)).float()
        }
    
    def _prefetch_embeddings(self, feature_ids: torch.Tensor):
        """Simulate prefetching embeddings to GPU cache."""
        unique_ids = feature_ids.unique().tolist()
        cache_misses = [i for i in unique_ids if i not in self.embedding_cache]
        # Simulate cache fill
        for idx in cache_misses:
            self.embedding_cache[idx] = True
        return len(cache_misses)
    
    def train_step(self, batch: Dict) -> Dict:
        """Execute one pipelined training step."""
        t0 = time.time()
        
        features = batch['features']
        labels = batch['labels']
        
        # Simulate pipeline stages
        cache_misses = self._prefetch_embeddings(features)
        
        logits = self.model(features)
        loss = F.binary_cross_entropy_with_logits(logits.squeeze(), labels)
        
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        step_time = time.time() - t0
        self.step_times.append(step_time)
        
        return {
            'loss': loss.item(),
            'step_time_ms': step_time * 1000,
            'cache_misses': cache_misses,
            'throughput': self.batch_size / step_time
        }


# Simple model for framework comparison
class SimpleDLRM(nn.Module):
    """Simplified DLRM for framework comparison."""
    def __init__(self, num_features: int, num_items: int, emb_dim: int = 16,
                 mlp_dims: List[int] = None):
        super().__init__()
        if mlp_dims is None:
            mlp_dims = [128, 64, 32]
        
        self.embedding = nn.EmbeddingBag(num_items, emb_dim, mode='sum')
        nn.init.xavier_uniform_(self.embedding.weight)
        
        input_dim = num_features * emb_dim
        layers = []
        for dim in mlp_dims:
            layers.extend([nn.Linear(input_dim, dim), nn.ReLU()])
            input_dim = dim
        layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)
        self.num_features = num_features
        self.emb_dim = emb_dim
    
    def forward(self, feature_ids: torch.Tensor) -> torch.Tensor:
        batch_size = feature_ids.size(0)
        embs = []
        for f in range(self.num_features):
            col = feature_ids[:, f]
            offsets = torch.arange(batch_size, dtype=torch.long)
            embs.append(self.embedding(col, offsets))
        x = torch.cat(embs, dim=1)
        return self.mlp(x)


# Benchmark the HugeCTR-style trainer
model = SimpleDLRM(num_features=10, num_items=50000, emb_dim=16)
trainer = HugeCTRStyleTrainer(model, batch_size=1024)

for step in range(50):
    batch = trainer._generate_batch(10, 50000)
    result = trainer.train_step(batch)
    if step % 10 == 0:
        print(f"Step {step}: loss={result['loss']:.4f}, "
              f"time={result['step_time_ms']:.1f}ms, "
              f"throughput={result['throughput']:.0f} samples/s")

## 4. DeepRec (Alibaba, 2022) & XDL (Alibaba, 2019)

### DeepRec
DeepRec is Alibaba's fork of TensorFlow with recommendation-specific optimizations:

- **Dynamic Embedding**: Hash-based embeddings that grow on demand (no fixed vocabulary)
- **Embedding Variable**: Optimized sparse parameter storage
- **Feature Column Plus**: Enhanced feature preprocessing
- **Smart Stage**: Automatic pipeline parallelism
- **AdaEmbed**: Adaptive embedding dimension

### XDL (eXtreme Deep Learning)
XDL focuses on extreme-scale training:

- Supports petabyte-scale embedding tables
- Custom PS architecture with SSD tiering
- Feature-level sparse optimization
- Serves Alibaba's Taobao, Tmall, and other e-commerce platforms

## 5. Merlin (NVIDIA, 2021)

Merlin is NVIDIA's end-to-end framework covering the full rec pipeline:

| Component | Purpose |
|-----------|---------|
| **NVTabular** | Feature engineering, preprocessing |
| **HugeCTR** | Training |
| **Triton** | Serving/inference |
| **Merlin Models** | High-level model library |

> **💡 Concept:** Merlin's key value proposition is the integrated pipeline — from raw data to served model — all GPU-accelerated. This eliminates the data transfer bottleneck between CPU preprocessing and GPU training.

In [ ]:
# Framework comparison: simulate different framework approaches

class FrameworkSimulator:
    """Simulate training characteristics of different frameworks."""
    
    def __init__(self, framework_name: str, model_config: Dict):
        self.name = framework_name
        self.config = model_config
        
        # Framework-specific overhead multipliers
        self.overheads = {
            'TorchRec': {
                'embedding_lookup': 1.0,
                'communication': 0.8,  # Optimized sharding
                'python_overhead': 1.2,  # PyTorch overhead
                'startup_time': 30.0,  # seconds
            },
            'HugeCTR': {
                'embedding_lookup': 0.6,  # GPU-native
                'communication': 0.9,
                'python_overhead': 0.3,  # Minimal Python
                'startup_time': 60.0,  # C++ compilation
            },
            'DeepRec': {
                'embedding_lookup': 0.9,
                'communication': 1.0,
                'python_overhead': 1.3,  # TF overhead
                'startup_time': 45.0,
            },
            'Merlin': {
                'embedding_lookup': 0.7,  # GPU-accelerated
                'communication': 0.8,
                'python_overhead': 0.8,  # Integrated pipeline
                'startup_time': 50.0,
            }
        }
    
    def estimate_step_time_ms(self, batch_size: int, num_workers: int = 1) -> float:
        """Estimate training step time for this framework."""
        overhead = self.overheads[self.name]
        
        # Base compute time (ms)
        base_compute = 5.0 * (batch_size / 1024)
        
        # Embedding lookup time
        num_tables = self.config.get('num_tables', 10)
        total_emb_rows = self.config.get('total_embedding_rows', 1e6)
        emb_time = 2.0 * num_tables * overhead['embedding_lookup']
        
        # Communication time (scales with workers)
        comm_time = 0.0
        if num_workers > 1:
            total_emb_mb = total_emb_rows * self.config.get('emb_dim', 64) * 4 / (1024**2)
            comm_time = total_emb_mb / 100 * overhead['communication']  # 100 MB/s effective
        
        # Python/framework overhead
        framework_overhead = 1.0 * overhead['python_overhead']
        
        return base_compute + emb_time + comm_time + framework_overhead
    
    def estimate_throughput(self, batch_size: int, num_workers: int = 1) -> float:
        """Samples per second."""
        step_time_s = self.estimate_step_time_ms(batch_size, num_workers) / 1000
        return batch_size * num_workers / step_time_s


# Compare frameworks across configurations
model_config = {
    'num_tables': 10,
    'total_embedding_rows': 10_000_000,
    'emb_dim': 64,
    'mlp_params': 500_000
}

frameworks = ['TorchRec', 'HugeCTR', 'DeepRec', 'Merlin']
worker_configs = [1, 2, 4, 8]

print(f"{'Framework':<12} {'1 GPU':>12} {'2 GPU':>12} {'4 GPU':>12} {'8 GPU':>12}")
print("-" * 60)

throughputs = defaultdict(list)
for fw in frameworks:
    sim = FrameworkSimulator(fw, model_config)
    row = f"{fw:<12}"
    for nw in worker_configs:
        tp = sim.estimate_throughput(4096, nw)
        throughputs[fw].append(tp)
        row += f" {tp:>10,.0f}/s"
    print(row)

In [ ]:
# Visualize framework comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'TorchRec': 'steelblue', 'HugeCTR': 'coral', 'DeepRec': 'green', 'Merlin': 'purple'}

# Throughput scaling
for fw in frameworks:
    axes[0].plot(worker_configs, throughputs[fw], 'o-', label=fw, 
                color=colors[fw], linewidth=2, markersize=8)

axes[0].set_xlabel('Number of GPUs', fontsize=12)
axes[0].set_ylabel('Throughput (samples/sec)', fontsize=12)
axes[0].set_title('Training Throughput Scaling', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Feature comparison radar chart (simplified as bar chart)
features = ['Ease of Use', 'Performance', 'Flexibility', 'Ecosystem', 'Scale', 'Serving']
scores = {
    'TorchRec': [4, 4, 5, 5, 5, 3],
    'HugeCTR': [2, 5, 2, 3, 4, 4],
    'DeepRec': [3, 3, 3, 3, 5, 3],
    'Merlin': [4, 4, 3, 4, 4, 5],
}

x = np.arange(len(features))
width = 0.2
for i, fw in enumerate(frameworks):
    axes[1].bar(x + i * width, scores[fw], width, label=fw, color=colors[fw], alpha=0.8)

axes[1].set_xticks(x + width * 1.5)
axes[1].set_xticklabels(features, rotation=30, ha='right')
axes[1].set_ylabel('Score (1-5)', fontsize=12)
axes[1].set_title('Framework Feature Comparison', fontsize=13)
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 6)

plt.tight_layout()
plt.show()

## 6. Building a TorchRec-Style Training Pipeline

Let us build a complete training pipeline that demonstrates TorchRec-style patterns:
- EmbeddingBagCollection for multi-table embeddings
- Automatic sharding planning
- Interaction architecture (dot product + MLP)

In [ ]:
class DLRMInteractionArch(nn.Module):
    """DLRM-style interaction architecture (dot product + over arch)."""
    
    def __init__(self, num_sparse_features: int, sparse_emb_dim: int,
                 num_dense_features: int, over_arch_dims: List[int]):
        super().__init__()
        
        # Dense feature bottom MLP
        self.dense_mlp = nn.Sequential(
            nn.Linear(num_dense_features, 64), nn.ReLU(),
            nn.Linear(64, sparse_emb_dim), nn.ReLU()
        )
        
        # Number of pairwise interactions: C(n+1, 2) where n = num_sparse + 1 (dense)
        n = num_sparse_features + 1
        num_interactions = n * (n - 1) // 2
        
        # Over arch (top MLP)
        input_dim = num_interactions + sparse_emb_dim
        layers = []
        for dim in over_arch_dims:
            layers.extend([nn.Linear(input_dim, dim), nn.ReLU()])
            input_dim = dim
        layers.append(nn.Linear(input_dim, 1))
        self.over_arch = nn.Sequential(*layers)
    
    def forward(self, dense_features: torch.Tensor,
                sparse_embeddings: List[torch.Tensor]) -> torch.Tensor:
        # Dense feature processing
        dense_emb = self.dense_mlp(dense_features)  # (B, D)
        
        # Concatenate all embeddings including dense
        all_embs = [dense_emb] + sparse_embeddings  # List of (B, D)
        all_embs_stacked = torch.stack(all_embs, dim=1)  # (B, N+1, D)
        
        # Pairwise dot product interactions
        interactions = torch.bmm(all_embs_stacked, all_embs_stacked.transpose(1, 2))  # (B, N+1, N+1)
        
        # Extract upper triangle (excluding diagonal)
        batch_size = interactions.size(0)
        n = interactions.size(1)
        triu_indices = torch.triu_indices(n, n, offset=1)
        flat_interactions = interactions[:, triu_indices[0], triu_indices[1]]  # (B, C(n,2))
        
        # Concatenate with dense embedding
        x = torch.cat([flat_interactions, dense_emb], dim=1)
        return self.over_arch(x)


class TorchRecStyleDLRM(nn.Module):
    """Complete DLRM model using TorchRec-style patterns."""
    
    def __init__(self, table_configs: List[EmbeddingTableConfig],
                 num_dense_features: int = 13, over_arch_dims: List[int] = None):
        super().__init__()
        if over_arch_dims is None:
            over_arch_dims = [128, 64]
        
        # Embedding collection
        self.ebc = EmbeddingBagCollection(table_configs)
        
        # Determine embedding dim (assume all same for simplicity)
        emb_dim = table_configs[0].embedding_dim
        num_sparse = len(table_configs)
        
        # Interaction architecture
        self.interaction = DLRMInteractionArch(
            num_sparse, emb_dim, num_dense_features, over_arch_dims
        )
        self.table_names = [t.name for t in table_configs]
    
    def forward(self, dense_features: torch.Tensor,
                sparse_features: Dict[str, torch.Tensor]) -> torch.Tensor:
        # Lookup embeddings
        emb_outputs = self.ebc(sparse_features)
        sparse_embs = [emb_outputs[name] for name in self.table_names]
        
        # Interaction + prediction
        return self.interaction(dense_features, sparse_embs)


# Create and train the model
table_configs = [
    EmbeddingTableConfig(f'feature_{i}', 10000, 16, [f'feature_{i}'], 'none')
    for i in range(10)
]

model = TorchRecStyleDLRM(table_configs, num_dense_features=13)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

# Training loop
losses = []
for step in range(200):
    batch_size = 512
    dense = torch.randn(batch_size, 13)
    sparse = {f'feature_{i}': torch.randint(0, 10000, (batch_size,)) for i in range(10)}
    labels = torch.randint(0, 2, (batch_size,)).float()
    
    logits = model(dense, sparse).squeeze()
    loss = F.binary_cross_entropy_with_logits(logits, labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    
    if step % 50 == 0:
        print(f"Step {step}: loss={loss.item():.4f}")

print(f"\nFinal loss: {np.mean(losses[-20:]):.4f}")

## 🏋️ Exercise 1: Build a TorchRec-Style Sharding Pipeline

Implement a sharded embedding lookup that distributes table lookups across simulated devices.

In [ ]:
# 🏋️ Exercise 1: Sharded embedding lookup

class ShardedEmbeddingBagCollection(nn.Module):
    def __init__(self, table_configs: List[EmbeddingTableConfig],
                 sharding_plans: List[ShardingPlan], num_devices: int):
        """
        Simulated sharded embedding collection.
        
        TODO:
        1. Parse sharding plans
        2. Create embedding tables on appropriate "devices"
        3. For row_wise: split table rows across devices
        4. For data_parallel: replicate tables
        5. For table_wise: place entire table on one device
        """
        super().__init__()
        pass
    
    def forward(self, features: Dict[str, torch.Tensor],
                requesting_device: int = 0) -> Dict[str, torch.Tensor]:
        """
        Look up embeddings, tracking cross-device communication.
        
        Returns:
            Dict of feature_name -> embedding tensor
        """
        # TODO: Implement with communication tracking
        pass


# Test:
# plans, _ = planner.plan(tables)
# sharded = ShardedEmbeddingBagCollection(tables, plans, num_devices=4)
# features = {'user_id': torch.randint(0, 100000, (256,)), ...}
# outputs = sharded(features)

## 🏋️ Exercise 2: Framework Selection Tool

Build a decision tool that recommends the best framework based on requirements.

In [ ]:
# 🏋️ Exercise 2: Framework selection tool

def recommend_framework(
    existing_stack: str,  # 'pytorch', 'tensorflow', 'none'
    model_size_gb: float,
    num_gpus: int,
    need_online_learning: bool,
    need_serving_integration: bool,
    team_size: str,  # 'small', 'medium', 'large'
    priority: str  # 'performance', 'flexibility', 'ease_of_use'
) -> Dict:
    """
    Recommend the best training framework based on requirements.
    
    TODO:
    1. Score each framework on the given criteria
    2. Weight scores by priority
    3. Return ranked recommendations with reasoning
    
    Returns:
        Dict with 'recommendation', 'scores', 'reasoning'
    """
    pass


# Test:
# result = recommend_framework(
#     existing_stack='pytorch',
#     model_size_gb=50,
#     num_gpus=8,
#     need_online_learning=True,
#     need_serving_integration=True,
#     team_size='medium',
#     priority='flexibility'
# )
# print(f"Recommended: {result['recommendation']}")
# print(f"Reasoning: {result['reasoning']}")

## 🏋️ Exercise 3: Benchmark Framework Patterns

Implement and benchmark three different embedding lookup patterns representing TorchRec, HugeCTR, and standard PyTorch approaches.

In [ ]:
# 🏋️ Exercise 3: Benchmark embedding lookup patterns

def benchmark_lookup_patterns(num_tables: int = 10, num_items: int = 100000,
                               emb_dim: int = 64, batch_size: int = 4096,
                               n_iterations: int = 100) -> Dict:
    """
    Benchmark different embedding lookup implementations:
    
    1. Standard: Individual nn.Embedding lookups
    2. Fused: Single concatenated embedding table with offsets
    3. Batched: Batch all lookups into one operation
    
    TODO:
    - Implement all three approaches
    - Measure forward and backward pass time
    - Return comparison results
    """
    pass


# results = benchmark_lookup_patterns()
# for name, metrics in results.items():
#     print(f"{name}: fwd={metrics['fwd_ms']:.2f}ms, bwd={metrics['bwd_ms']:.2f}ms")

## Summary

| Framework | Best For | Avoid When |
|-----------|----------|------------|
| **TorchRec** | PyTorch shops, flexible architectures | Need TF ecosystem |
| **HugeCTR** | Maximum throughput, standard CTR models | Need custom architectures |
| **DeepRec** | TF shops, Alibaba-style features | Small scale |
| **Merlin** | End-to-end GPU pipeline | Don't have NVIDIA GPUs |
| **XDL** | Extreme scale (PB embeddings) | Small/medium scale |

### Key References
- Ivchenko et al., "TorchRec: a PyTorch Domain Library for Recommendation Systems" (2022, Meta)
- NVIDIA, "HugeCTR: High-Performance Click-Through Rate Estimation Training" (2020)
- Alibaba, "DeepRec: A TensorFlow-based Recommendation Engine" (2022)
- NVIDIA, "Merlin: A GPU-Accelerated Recommendation Framework" (2021)
- Jiang et al., "XDL: An Industrial Deep Learning Framework for High-dimensional Sparse Data" (2019, Alibaba)

### Next Steps
In the next notebook (9.7), we will explore hyperparameter optimization techniques specifically designed for recommendation models.